# Question 2: Bootstrapping a ratio of means

Two filling machines are checked by weighing eight containers from each. The quantity of
interest is the ratio of the two mean fill weights, $\hat{R} = \bar{x}_A / \bar{x}_B$.

You will compute a standard error two ways, once from a formula and once by resampling, and
then find a standard error for a quantity that has no formula.

Save your own copy first (File > Save a copy in Drive). Then paste the INPUTS block from the
course page over the INPUTS cell below, replacing everything in that cell.

Run the cells in order and fill in every TODO. Use only `numpy` and `matplotlib`.

In [ ]:
# ===== INPUTS =====
# Replace this whole cell with the INPUTS block from the course page.
GROUP_A = None
GROUP_B = None
N_RESAMPLES = 4000
SEED_A = 20260912
SEED_B = 20260913
# ==================

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if GROUP_A is None or GROUP_B is None:
    raise ValueError("Paste the INPUTS block from the course page over the INPUTS cell, "
                     "run that cell, then run this one again.")

A = np.asarray(GROUP_A, dtype=float)
B = np.asarray(GROUP_B, dtype=float)
print(f"group A: n = {len(A)}, mean = {A.mean():.4f}")
print(f"group B: n = {len(B)}, mean = {B.mean():.4f}")
print(f"ratio of means = {A.mean() / B.mean():.4f}")

### TODO 1: the standard error from a formula

Complete `se_of_mean` so that it returns the standard error of the sample mean of `x`:

$$ \mathrm{se}(\bar{x}) \; = \; \frac{s}{\sqrt{n}} $$

where $n$ is the number of observations and $s$ is the sample standard deviation with
divisor $n - 1$.

In [ ]:
def se_of_mean(x):
    """Standard error of the sample mean, from the textbook formula."""
    # TODO 1: return s / sqrt(n), with s the sample standard deviation (divisor n - 1)
    return ...

### TODO 2: the same standard error by resampling

Complete `boot_means` so that it returns an array of `n_resamples` bootstrap resample means
of `x`.

Create the generator once, with `np.random.RandomState(seed)`. Then, in each of the
`n_resamples` iterations, draw `len(x)` values from `x` with replacement and store their
mean. Use that generator's `.choice` method rather than `np.random.choice`, so your result is
reproducible. Its signature is

```
rs.choice(a, size=None, replace=True, p=None)
```

`a` is what you draw from, `size` is how many values you draw, and `replace` says whether the
same value may be drawn more than once. `p` would weight the draw, and is not needed here.

In [ ]:
def boot_means(x, n_resamples, seed):
    """An array of n_resamples bootstrap resample means of x."""
    rs = np.random.RandomState(seed)
    # TODO 2: return an array of n_resamples resample means, each from a draw of
    #         len(x) values from x with replacement
    return ...

### TODO 3: a standard error with no formula

There is a textbook formula for the standard error of a mean, but there is none for the
standard error of the ratio $\hat{R} = \bar{x}_A / \bar{x}_B$. Resampling
gives a standard error for complicated functions of the data that have no closed-form
formula.

Complete `boot_se_ratio` so that it:

1. resamples each group with its own generator, group $A$ with `seed_a` and group $B$
   with `seed_b`, so the order you do them in cannot change the answer;
2. computes the ratio of the $i$th resample mean of group $A$ and group $B$, obtaining an
   array of ratios of length `n_resamples`;
3. returns the standard deviation of those `n_resamples` ratios, using `.std()` as it comes.

You may call `boot_means`.

In [ ]:
def boot_se_ratio(a, b, n_resamples, seed_a, seed_b):
    """Bootstrap standard error of the ratio of the two group means."""
    # TODO 3: resample each group with its own seed, form the ratio of the two
    #         resample means, and return the standard deviation of those ratios
    return ...


se_formula = se_of_mean(A)
se_boot_mean = boot_means(A, N_RESAMPLES, SEED_A).std()
se_boot_ratio = boot_se_ratio(A, B, N_RESAMPLES, SEED_A, SEED_B)

print("mean of group A")
print(f"  standard error, formula     = {se_formula:.5f}")
print(f"  standard error, bootstrap   = {se_boot_mean:.5f}")
print()
print("ratio of the two means")
print(f"  ratio                       = {A.mean() / B.mean():.5f}")
print(f"  standard error, bootstrap   = {se_boot_ratio:.5f}")
print(f"  standard error, formula     = none exists")

### See the resampling distributions

The left panel is the distribution of the resample means of group $A$, with the formula
standard error marked either side of the sample mean. The right panel is the distribution of
the resample ratios. Only the left panel has a formula to check against.

In [ ]:
boot_a = boot_means(A, N_RESAMPLES, SEED_A)
boot_r = boot_a / boot_means(B, N_RESAMPLES, SEED_B)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(boot_a, bins=40, color="#9cc3e6", edgecolor="white")
axes[0].axvline(A.mean(), color="red", linestyle="--", label=f"sample mean = {A.mean():.2f}")
for sign in (-1, 1):
    axes[0].axvline(A.mean() + sign * se_formula, color="black", linestyle=":",
                    label="formula SE" if sign == 1 else None)
axes[0].set(xlabel="resample mean of group A", ylabel="count",
            title="Resample means, with the formula SE")
axes[0].legend()

axes[1].hist(boot_r, bins=40, color="#c6b5e6", edgecolor="white")
axes[1].axvline(A.mean() / B.mean(), color="red", linestyle="--",
                label=f"sample ratio = {A.mean() / B.mean():.3f}")
axes[1].set(xlabel="resample ratio of means", ylabel="count",
            title="Resample ratios, with no formula to mark")
axes[1].legend()
plt.tight_layout()
plt.show()

### How stable is a bootstrap standard error?

A bootstrap standard error is itself an estimate, computed from a finite number of resamples,
so it carries its own noise. The cell below recomputes the same quantity with twelve
different seeds at each of three resample counts, and reports the spread across those twelve
runs. That spread falls roughly like $1/\sqrt{n_{\text{resamples}}}$.

The formula standard error is printed for comparison: it does not depend on resampling at
all, so it has no spread whatsoever.

In [ ]:
print(f"formula SE of the mean of A: {se_formula:.5f}  (no resampling, no spread)")
print()
print("bootstrap SE of the same mean, recomputed with twelve different seeds:")
print(f"{'resamples':>10}   {'lowest':>8}   {'highest':>8}   {'spread':>8}")
for n_res in (100, 1000, 10000):
    runs = [boot_means(A, n_res, SEED_A + shift).std() for shift in range(12)]
    print(f"{n_res:>10}   {min(runs):>8.4f}   {max(runs):>8.4f}   {np.std(runs):>8.5f}")

### Report your results

Run the cell below and enter R1, R2 and R3 on the course page. All three are given to five
decimal places, because R3 is a small number.

In [ ]:
print("=" * 60)
print("QUESTION 2: REPORT VALUES")
print("=" * 60)
print(f"R1. Formula SE, mean of A (5 dec):      {se_formula:.5f}")
print(f"R2. Bootstrap SE, mean of A (5 dec):    {se_boot_mean:.5f}")
print(f"R3. Bootstrap SE, ratio A/B (5 dec):    {se_boot_ratio:.5f}")

### Multiple-choice A

*This also appears on the course page, where you answer it. The options there may be in a different order.*

Your notebook printed a formula and bootstrap standard error for the mean
of group $A$, and a bootstrap standard error for the ratio of the two means. Which statement
is best supported?

1. The bootstrap is the more accurate of the two for the mean, because it uses the data several thousand times rather than once.
2. The standard error of the ratio could have been obtained as the ratio of the two groups' formula standard errors.
3. The bootstrap and the formula should agree exactly for the mean, so the gap printed is a coding error.
4. For the mean the bootstrap approximately matches what the formula already gives; for the ratio no formula was available, and the bootstrap supplies the only estimate.

### Counterfactual

`SEED_SHIFT` changes which resamples get drawn. The two samples of eight weights are
untouched; only the random draws differ.

Change `SEED_SHIFT` from `0` to `2` in the cell below, re-run that cell only, and compare
its output with the values you reported above. The multiple-choice question on the course
page asks what changed.

In [ ]:
SEED_SHIFT = 0     # <-- change to 2 and re-run this cell

print(f"SEED_SHIFT = {SEED_SHIFT}")
print(f"  formula SE, mean of A    = {se_of_mean(A):.5f}")
print(f"  bootstrap SE, mean of A  = "
      f"{boot_means(A, N_RESAMPLES, SEED_A + SEED_SHIFT).std():.5f}")
print(f"  bootstrap SE, ratio A/B  = "
      f"{boot_se_ratio(A, B, N_RESAMPLES, SEED_A + SEED_SHIFT, SEED_B + SEED_SHIFT):.5f}")

### Multiple-choice B

*This also appears on the course page, where you answer it. The options there may be in a different order.*

You changed `SEED_SHIFT` from `0` to `2` and re-ran the last cell. The two samples of eight
weights were not touched; only which resamples were drawn changed. Which statement matches
the output?

1. The formula standard error is unchanged, and both bootstrap standard errors change a little.
2. All three values are unchanged, because the eight weights did not change.
3. All three values change, because all three are computed from resamples.
4. The formula standard error changes and the two bootstrap values do not, because only the formula is computed from the sample.